<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 14 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Separate Dashboard and Ad Hoc Resources</div>
</div>

The dashboard of Module 11 and ad hoc analysis share one Workload Group, and a long scan slows the dashboard down. You give each workload a group of its own, watch ad hoc queries run, wait, be rejected, and bypass the queue, and tell a queue timeout from an execution timeout. You route a temporary ordinary user to the dashboard group with its account's default, and see Doris refuse it a group that it may not use. You reject an unfiltered scan with a SQL Block Rule, remove everything in the order that its dependencies require, and check a resource plan before you apply it. Plan about 45 minutes.

**Module 14: Resource Isolation and Query Protection** · [Lesson](course.md) · [Quiz](quiz14_resource_isolation.ipynb) · [Course README](../../README.md)

## Before you begin

This Lab needs only the course sandbox: no earlier Lab's tables. It rebuilds the table `orders_monthly_l3` in your lab database.

Workload Groups, roles, users, and SQL Block Rules belong to the whole Doris cluster, not to your lab database. The Lab creates them with fixed names, the groups `course_dashboard_l3` and `course_adhoc_l3`, the role `course_dashboard_reader_l3`, and the rule `course_scan_guard_l3`, and in section 6 a temporary ordinary user with a random name, whose account default it sets; the practice creates `course_plan_dashboard_l3` and `course_plan_adhoc_l3`. Each section removes what it creates, section 8 the role and the groups, and step 2 removes what an interrupted run left. The group `normal` and root's account are not changed.

The SQL Block Rule of section 7 applies to every session of the sandbox, root's too, while it exists: the cell creates it, tests it, and drops it in `finally`. If that cell is interrupted or the kernel restarts, run the connection cell and step 2 again before anything else. The slow queries use `sleep()`, and the sessions turn the SQL cache off so that each query really runs. The sandbox has one FE, one BE, and no CPU cgroup: CPU limits are not enforced here.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.access import temporary_user
from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect
from dw_course.streaming import wait_for
from dw_course.ui import show_records, show_sql
from dw_course.workload import BackgroundQuery, group_load, run_query, show_outcomes, wait_for_group

lab = connect_sandbox(module=14)

DASHBOARD = "course_dashboard_l3"
ADHOC = "course_adhoc_l3"

## 1. Which group does a query use now?

Doris picks a query's Workload Group in this order: a `SET_VAR(workload_group=...)` hint in the query, the session's `workload_group`, the account's `default_workload_group`, and otherwise `normal`. This Lab uses no hints. The cell shows root's session value and account default and the groups that exist, and keeps `normal`'s settings to compare them at the end.

In [ ]:
lab.sql("SELECT @@workload_group AS session_workload_group", title="The session's workload_group")
lab.sql("SHOW PROPERTY LIKE 'default_workload_group'", title="The account's default group")
lab.sql("SHOW WORKLOAD GROUPS", title="The Workload Groups that exist",
        columns=["Name", "max_concurrency", "max_queue_size", "queue_timeout", "max_memory_percent", "running_query_num", "waiting_query_num"])
NORMAL_BEFORE = lab.query("SELECT * FROM information_schema.workload_groups WHERE NAME = 'normal'")
expect((lab.query("SELECT @@workload_group"), lab.query("SHOW PROPERTY LIKE 'default_workload_group'")),
       ([("",)], [("default_workload_group", "normal")]), title="Root's queries run in normal")

**Expected result**

An empty session `workload_group`, the account default `normal`, and the group `normal` with `max_concurrency` 2147483647, `max_queue_size` 0, `queue_timeout` 0, and `max_memory_percent` 100%: no practical limit, no queue. A green check: root's queries run in `normal`. If a course group is listed too, an earlier run was interrupted; step 2 removes it.

## 2. Prepare the table and remove leftovers

Removing the Lab's cluster-wide objects from an earlier run needs an order, because each one can hold another: a user whose default is a course group keeps the group from being dropped, and a global SQL Block Rule would block the full scan below. The cell drops leftover temporary users of section 6, then the rule, the role, and the groups, and rebuilds `orders_monthly_l3` with the monthly partitions of Module 13 and one order per month.

In [ ]:
import pymysql

with lab.connection.cursor(pymysql.cursors.DictCursor) as cursor:
    cursor.execute("SHOW ALL GRANTS")
    leftover_users = [row["UserIdentity"] for row in cursor.fetchall() if row["UserIdentity"].startswith("'course_routing_l3_")]
for identity in leftover_users:
    lab.execute(f"DROP USER IF EXISTS {identity}")
lab.execute("DROP SQL_BLOCK_RULE IF EXISTS course_scan_guard_l3")
lab.execute("DROP ROLE IF EXISTS course_dashboard_reader_l3")
lab.execute("DROP WORKLOAD GROUP IF EXISTS course_dashboard_l3")
lab.execute("DROP WORKLOAD GROUP IF EXISTS course_adhoc_l3")

lab.execute("DROP TABLE IF EXISTS orders_monthly_l3")
lab.execute("""
CREATE TABLE orders_monthly_l3 (
    order_date DATE NOT NULL,
    order_id BIGINT NOT NULL,
    amount DECIMAL(18,2) NOT NULL
)
DUPLICATE KEY(order_date, order_id)
PARTITION BY RANGE(order_date) (
    PARTITION p202501 VALUES LESS THAN ("2025-02-01"),
    PARTITION p202502 VALUES LESS THAN ("2025-03-01"),
    PARTITION p202503 VALUES LESS THAN ("2025-04-01")
)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.insert("orders_monthly_l3", ["order_date", "order_id", "amount"], [("2025-01-15", 140001, 100), ("2025-02-15", 140002, 200), ("2025-03-15", 140003, 300)])
lab.sql("SELECT * FROM orders_monthly_l3 ORDER BY order_date, order_id", title="The monthly table")
expect(lab.query("SELECT CAST(order_date AS STRING), order_id, amount FROM orders_monthly_l3 ORDER BY order_date, order_id"),
       [("2025-01-15", 140001, "100.00"), ("2025-02-15", 140002, "200.00"), ("2025-03-15", 140003, "300.00")],
       title="One order in each of three months")

**Expected result**

Three orders, 140001 on 2025-01-15 (100.00), 140002 on 2025-02-15 (200.00), and 140003 on 2025-03-15 (300.00), one in each partition, and a green check. Section 4's slow query reads only March's row; section 7's unfiltered query scans all three partitions.

## 3. Create the dashboard and ad hoc groups

The settings are those of the runbook in lesson section 14.6. Ad hoc concurrency and queue capacity are both 1, making queuing and rejection visible with a few requests.

The Backend (BE) has no CPU cgroup configured, so max_cpu_percent is not enforced in this sandbox. Grant only dashboard-group USAGE_PRIV to the course role.

In [ ]:
lab.execute("""
CREATE WORKLOAD GROUP course_dashboard_l3
PROPERTIES (
    "max_concurrency" = "8",
    "max_queue_size" = "20",
    "queue_timeout" = "10000",
    "max_memory_percent" = "50"
)
""")
lab.execute("""
CREATE WORKLOAD GROUP course_adhoc_l3
PROPERTIES (
    "max_concurrency" = "1",
    "max_queue_size" = "1",
    "queue_timeout" = "30000",
    "max_memory_percent" = "30",
    "max_cpu_percent" = "30"
)
""")
lab.execute("CREATE ROLE course_dashboard_reader_l3")
lab.execute("GRANT USAGE_PRIV ON WORKLOAD GROUP 'course_dashboard_l3' TO ROLE 'course_dashboard_reader_l3'")
lab.sql("SHOW WORKLOAD GROUPS LIKE 'course%'", title="The two course Workload Groups",
        columns=["Name", "max_concurrency", "max_queue_size", "queue_timeout", "max_memory_percent", "max_cpu_percent"])
expect(
    lab.query("""
SELECT NAME, MAX_CONCURRENCY, MAX_QUEUE_SIZE, QUEUE_TIMEOUT, MAX_MEMORY_PERCENT
FROM information_schema.workload_groups
WHERE NAME IN ('course_dashboard_l3', 'course_adhoc_l3')
ORDER BY NAME
"""),
    [
        ("course_adhoc_l3", 1, 1, 30000, "30%"),
        ("course_dashboard_l3", 8, 20, 10000, "50%"),
    ],
    title="Limits for both groups",
)
expect(
    lab.query("""
SELECT GRANTEE, WORKLOAD_GROUP_NAME, PRIVILEGE_TYPE
FROM information_schema.workload_group_privileges
WHERE WORKLOAD_GROUP_NAME IN ('course_dashboard_l3', 'course_adhoc_l3')
ORDER BY GRANTEE, WORKLOAD_GROUP_NAME
"""),
    [("course_dashboard_reader_l3", "course_dashboard_l3", "Usage_priv")],
    title="Course role has usage only on the dashboard group",
)

**Expected result**

The two groups: `course_adhoc_l3` with concurrency 1, queue 1, a wait of 30000 ms, 30% memory, and 30% CPU; `course_dashboard_l3` with 8, 20, 10000 ms, 50% memory, and 100% CPU. Two green checks: the limits, and the role's `Usage_priv` on the dashboard group alone.

`USAGE_PRIV` allows a role's members to use the group; it does not send their queries there. Section 6 routes a user with its account's default. The sandbox has no CPU cgroup, so the 30% CPU of the ad hoc group is a setting that nothing enforces here.

## 4. Submit five queries: run, queue, or reject?

sleep() simulates a slow scan of March's single row, so sleep(6) takes about six seconds. Each query opens a separate session, disables SQL Cache, sets its session variables, then runs.

A cache hit would return a previous result without entering the queue, so disabling the cache is necessary for this experiment.

- A occupies the ad hoc group's only running slot.
- B queues behind A.
- C enters the same group after the queue is full and should be rejected.
- D uses the dashboard group's own slot.
- E uses the ad hoc group with bypass_workload_group enabled.

Before submitting each dependent request, wait for the expected running and waiting counts to avoid a timing race.

In [ ]:
SLOW = "SELECT order_id, sleep({seconds}) AS slow_scan FROM orders_monthly_l3 WHERE order_date >= '2025-03-01'"

a = BackgroundQuery(lab, "A ad hoc", SLOW.format(seconds=6), workload_group=ADHOC)
wait_for_group(lab, ADHOC, running=1, waiting=0)
b = BackgroundQuery(lab, "B ad hoc", SLOW.format(seconds=1), workload_group=ADHOC)
wait_for_group(lab, ADHOC, running=1, waiting=1)

lab.sql("""
SELECT q.QUERY_STATUS, g.NAME AS workload_group, q.QUEUE_START_TIME, q.QUEUE_END_TIME
FROM information_schema.active_queries q
JOIN information_schema.workload_groups g ON q.WORKLOAD_GROUP_ID = g.ID
WHERE g.NAME = 'course_adhoc_l3'
ORDER BY q.QUERY_STATUS
""", title="A is running; B is queued");

**Expected result**

Two rows of `active_queries` for the ad hoc group: one `RUNNING`, A, with both queue times set, and one `WAIT_IN_QUEUE`, B, with a `QUEUE_START_TIME` and no `QUEUE_END_TIME`: it has not got a slot yet.

In [ ]:
c = run_query(lab, "C ad hoc", SLOW.format(seconds=1), workload_group=ADHOC)
d = run_query(lab, "D dashboard", SLOW.format(seconds=1), workload_group=DASHBOARD)
e = run_query(lab, "E queue bypass", SLOW.format(seconds=1), workload_group=ADHOC, bypass_workload_group=True)
load_while_a_runs = group_load(lab, ADHOC)

a_outcome, b_outcome = a.result(), b.result()
show_outcomes("Outcomes of five queries", [a_outcome, b_outcome, c, d, e])
wait_for_group(lab, ADHOC, running=0, waiting=0)
expect(
    [(o.label, o.result) for o in (a_outcome, b_outcome, c, d, e)],
    [
        ("A ad hoc", "OK"),
        ("B ad hoc", "OK"),
        ("C ad hoc", "query waiting queue is full, queue capacity=1, waiting num=1"),
        ("D dashboard", "OK"),
        ("E queue bypass", "OK"),
    ],
    title="Execution, queuing, rejection, and bypass",
)
expect(load_while_a_runs, {"running": 1, "waiting": 1}, title="Ad hoc counters after C, D, and E finish")
expect(b_outcome.seconds > d.seconds + 3, True, title="B elapsed time includes queue wait")

**Expected result**

The outcomes of the five queries: A OK after about 6 seconds; B OK after about 7, because it waited for A; C rejected at once with `query waiting queue is full, queue capacity=1, waiting num=1`; D, in the dashboard group, OK after about 1 second; E, with the queue bypassed, OK after about 1 second. Three green checks: the five outcomes, the ad hoc counters of one running and one waiting query while C, D, and E ran, and B's time including its wait.

C was rejected because B held the only place in the queue. D did not wait: the dashboard group has slots of its own. E ran in the same group as C by setting `bypass_workload_group`, which any user can set for their session: the queue is not an isolation boundary, while the group's usage check and the BE's memory limits still apply.

## 5. Limit the queue wait and the execution separately

queue_timeout limits waiting; query_timeout limits execution.

First shorten the ad hoc group's wait cap to one second and repeat the occupied-slot test. Then run F, a four-second query, with a two-second execution timeout. F does not select a group and uses normal.

This step changes the ad hoc wait cap. To repeat step 3 with its original settings, rerun from step 1.

In [ ]:
lab.execute('ALTER WORKLOAD GROUP course_adhoc_l3 PROPERTIES ("queue_timeout" = "1000")')

a = BackgroundQuery(lab, "A ad hoc", SLOW.format(seconds=3), workload_group=ADHOC)
wait_for_group(lab, ADHOC, running=1, waiting=0)
b = run_query(lab, "B queue timeout", SLOW.format(seconds=1), workload_group=ADHOC)
f = run_query(lab, "F execution timeout", SLOW.format(seconds=4), query_timeout=2)

a_outcome = a.result()
show_outcomes("Queue and execution timeouts", [a_outcome, b, f])
wait_for_group(lab, ADHOC, running=0, waiting=0)
expect(
    [(o.label, o.result) for o in (a_outcome, b, f)],
    [
        ("A ad hoc", "OK"),
        ("B queue timeout", "query queue timeout, timeout: 1000 ms"),
        ("F execution timeout", "query timeout"),
    ],
    title="Two distinct timeout errors",
)
expect(b.seconds < 2, True, title="B stops waiting after about one second")
expect(1.5 <= f.seconds < 4, True, title="F is cancelled after about two seconds of execution")

**Expected result**

The outcomes: A OK after about 3 seconds; B fails after about 1 second with `query queue timeout, timeout: 1000 ms`; F, in `normal`, fails after about 2 seconds with `query timeout`. Three green checks: the two errors, B's wait of under 2 seconds, and F's 1.5 to 4 seconds.

B never got a slot, so it never ran on the BE. F ran and used resources until Doris cancelled it. A user can raise their session's `query_timeout`; an administrator's account property of the same name takes precedence and cannot be changed by the user.

## 6. Route an ordinary user with its account's default

Root ran every query so far. A dashboard's account should land in the dashboard group without setting anything in its sessions: give its role `USAGE_PRIV` on the group, and set the account's `default_workload_group`. The cell gives the role read access to the table, creates a temporary user with the role, and sets its default with `SET PROPERTY FOR`, an administrator's statement.

As that user, it runs a slow query and reads in `active_queries` which group the query runs in. Then the user sets its session to the ad hoc group, which its role may not use. Predict what Doris answers. In `finally`, the cell sets the user's default back to `normal` and drops the user: a group that is still some account's default cannot be dropped.

In [ ]:
lab.execute(f"GRANT SELECT_PRIV ON internal.{lab.database}.orders_monthly_l3 TO ROLE 'course_dashboard_reader_l3'")
SLOW = "SELECT order_id, sleep({seconds}) AS slow_scan FROM orders_monthly_l3 WHERE order_date >= '2025-03-01'"
with temporary_user(lab, "course_dashboard_reader_l3", prefix="course_routing_l3") as user:
    default_sql = f"SET PROPERTY FOR '{user.name}' 'default_workload_group' = '{DASHBOARD}'"
    show_sql("Route the user's queries to the dashboard group", default_sql + ";")
    lab.execute(default_sql)
    try:
        g = BackgroundQuery(lab, "G as the dashboard user", SLOW.format(seconds=3), user=user)
        running = wait_for(lambda: lab.query("""SELECT g.NAME, q.QUERY_STATUS FROM information_schema.active_queries q
            JOIN information_schema.workload_groups g ON q.WORKLOAD_GROUP_ID = g.ID WHERE q.USER = %s""", (user.name,)),
            bool, description="Find the user's running query", timeout=10)
        h = run_query(lab, "H as the dashboard user, in the ad hoc group", SLOW.format(seconds=1), user=user, workload_group=ADHOC)
        show_outcomes("The dashboard user's two queries", [g.result(), h])
        show_records("Where the user's query ran", [{"user": user.name, "workload_group": running[0][0], "status": running[0][1]}])
        expect(running[0][0], DASHBOARD, title="The user's query runs in its default group")
        expect(h.result.startswith(f"Access denied; you need (at least one of) the USAGE/ADMIN privilege(s) to use workload group '{ADHOC}'"),
               True, title="The user may not use the ad hoc group")
    finally:
        lab.execute(f"SET PROPERTY FOR '{user.name}' 'default_workload_group' = 'normal'")

**Expected result**

The statements that create the user and set its default, then the outcomes: G OK after about 3 seconds; H refused at once with `Access denied; you need (at least one of) the USAGE/ADMIN privilege(s) to use workload group 'course_adhoc_l3'. User: 'course_routing_l3_<random>'@'%'`. The user's query ran in `course_dashboard_l3`, `RUNNING`. Two green checks.

The user named no group: its account's default routed the query. Doris checks the privilege when the query runs, not when the session variable is set: the `SET workload_group` succeeded, the query failed. With the default set and no `USAGE_PRIV`, every query of the account would fail the same way, so set both together.

## 7. Reject a full scan with a SQL Block Rule

The global rule permits scans of at most two partitions. An unrestricted query scans three and should fail; a March-only query prunes to one and succeeds.

The rule affects all sandbox sessions, including root. Create, test, and delete it within the same cell, using try/finally for cleanup. An interruption or a kernel restart can still leave it behind; step 2 removes it.

Use SUM(amount), because an unfiltered COUNT(*) may be answered without scanning partitions, as lesson section 14.5 explains.

In [ ]:
lab.execute("""
CREATE SQL_BLOCK_RULE course_scan_guard_l3
PROPERTIES ("partition_num" = "2", "global" = "true", "enable" = "true")
""")
try:
    lab.sql("SHOW SQL_BLOCK_RULE FOR course_scan_guard_l3", title="Scan rule")
    full_scan = run_query(lab, "Full-table SUM", "SELECT SUM(amount) FROM orders_monthly_l3")
    one_month = run_query(lab, "March only", "SELECT SUM(amount) FROM orders_monthly_l3 WHERE order_date >= '2025-03-01'")
finally:
    lab.execute("DROP SQL_BLOCK_RULE IF EXISTS course_scan_guard_l3")

show_outcomes("Scan-rule outcomes", [full_scan, one_month])
expect(
    [(o.label, o.result) for o in (full_scan, one_month)],
    [
        ("Full-table SUM", "sql hits sql block rule: course_scan_guard_l3, reach partition_num : 2"),
        ("March only", "OK"),
    ],
    title="Scan rule rejects the full scan",
)
expect(
    [row for row in lab.query("SHOW SQL_BLOCK_RULE") if row[0] == "course_scan_guard_l3"],
    [],
    title="Global rule has been removed",
)

**Expected result**

The rule, with `PartitionNum` 2, `Global` true, and `Enable` true; the outcomes: the full-table `SUM` rejected with `sql hits sql block rule: course_scan_guard_l3, reach partition_num : 2`, the March query OK. Two green checks: the rule rejected the full scan, and it is gone.

The rejected query never ran. The 2 in the message is the rule's limit, not the three partitions that the scan needed; the March query read one. The rule existed for less than a second and applied to every session of the sandbox meanwhile; associate a rule with the analysts' accounts with `sql_block_rules`, an administrator's account property, when it should apply to them alone.

## 8. Revoke access, and remove the role and the groups

Check the dependencies first: no Workload Policy references the groups, and section 6 set its user's default back to `normal` and dropped the user.

Then revoke `USAGE_PRIV`, drop the role, and drop both groups: a group's grants outlive the group, and a group recreated later with the same name would get them back. Keep orders_monthly_l3 for further inspection; step 2 rebuilds it on the next run.

In [ ]:
expect(
    lab.query("""
SELECT COUNT(*) FROM information_schema.workload_policy
WHERE WORKLOAD_GROUP IN ('course_dashboard_l3', 'course_adhoc_l3')
"""),
    [(0,)],
    title="No Workload Policy references the course groups",
)
lab.execute("REVOKE USAGE_PRIV ON WORKLOAD GROUP 'course_dashboard_l3' FROM ROLE 'course_dashboard_reader_l3'")
lab.execute("DROP ROLE IF EXISTS course_dashboard_reader_l3")
lab.execute("DROP WORKLOAD GROUP IF EXISTS course_dashboard_l3")
lab.execute("DROP WORKLOAD GROUP IF EXISTS course_adhoc_l3")
lab.sql("SHOW WORKLOAD GROUPS", title="The Workload Groups after cleanup",
        columns=["Name", "max_concurrency", "max_queue_size", "queue_timeout", "max_memory_percent"])
expect(
    lab.query("""
SELECT
    (SELECT COUNT(*) FROM information_schema.workload_groups
     WHERE NAME IN ('course_dashboard_l3', 'course_adhoc_l3')),
    (SELECT COUNT(*) FROM information_schema.workload_group_privileges
     WHERE WORKLOAD_GROUP_NAME IN ('course_dashboard_l3', 'course_adhoc_l3'))
"""),
    [(0, 0)],
    title="Course groups and usage grants have been removed",
)
expect([row[0] for row in lab.query("SHOW ROLES") if row[0] == "course_dashboard_reader_l3"], [], title="Course role has been removed")
expect(lab.query("SELECT * FROM information_schema.workload_groups WHERE NAME = 'normal'"), NORMAL_BEFORE, title="normal remains unchanged")

**Expected result**

A green check, no Workload Policy references the groups; the groups that remain, `normal` alone; then three green checks: the course groups and their usage grants are gone, the role is gone, and `normal` has the settings that step 1 kept. `orders_monthly_l3` stays for you to query; step 2 rebuilds it on the next run.

## 9. Practice: check a resource plan before you apply it

A dashboard goes live on a cluster with three FEs. Five BI accounts send about 20 queries at once when the dashboard refreshes; three analysts send irregular scans. Write the plan as a dictionary: for each group its concurrency, queue, and wait in milliseconds. Check it in Python first: on three evenly used FEs, the dashboard's slots must hold the 20 queries, and the ad hoc queue must stay small. Then create the two groups under trial names, `course_plan_dashboard_l3` and `course_plan_adhoc_l3`, read their settings back from `information_schema.workload_groups`, compare them with the plan, and drop them.

In [ ]:
# Your code here

The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
plan = {
    "course_plan_dashboard_l3": {"max_concurrency": 8, "max_queue_size": 20, "queue_timeout": 10000},
    "course_plan_adhoc_l3": {"max_concurrency": 1, "max_queue_size": 2, "queue_timeout": 30000},
}
FES, PEAK = 3, 20
expect((plan["course_plan_dashboard_l3"]["max_concurrency"] * FES >= PEAK, plan["course_plan_adhoc_l3"]["max_queue_size"] <= 5),
       (True, True), title="The plan holds the peak on three FEs and keeps the ad hoc queue small")
try:
    for name, settings in plan.items():
        properties = ", ".join(f'"{key}" = "{value}"' for key, value in settings.items())
        lab.execute(f"DROP WORKLOAD GROUP IF EXISTS {name}")
        lab.execute(f"CREATE WORKLOAD GROUP {name} PROPERTIES ({properties})")
    applied = lab.sql("""SELECT NAME, MAX_CONCURRENCY, MAX_QUEUE_SIZE, QUEUE_TIMEOUT FROM information_schema.workload_groups
                         WHERE NAME LIKE 'course_plan_%' ORDER BY NAME""", title="The groups as Doris applied them")
    expect([tuple(row) for row in applied.itertuples(index=False)],
           sorted((name, s["max_concurrency"], s["max_queue_size"], s["queue_timeout"]) for name, s in plan.items()),
           title="Doris applied the plan as written")
finally:
    for name in plan:
        lab.execute(f"DROP WORKLOAD GROUP IF EXISTS {name}")

**Expected result**

A green check: 8 slots on each of three FEs give 24 for a peak of 20, and the ad hoc queue holds 2. The two groups as Doris applied them: `course_plan_adhoc_l3` with 1, 2, and 30000, and `course_plan_dashboard_l3` with 8, 20, and 10000; a green check; then the cell drops both.

The queue is per FE: 24 slots exist only if the connections are spread over the three FEs; if they all go to one FE, its 8 slots and 20 places in the queue are what the dashboard has. Check the connection distribution before you trust the sum, and route the accounts with `USAGE_PRIV` and `default_workload_group` as section 6 did.

## You are done when

- you can say in which order Doris picks a query's group, and which group root's and the dashboard user's queries ran in;
- you can predict from a group's concurrency, queue, and wait whether a query runs, waits, or is rejected, and say why `bypass_workload_group` is not an isolation boundary;
- you can tell `query queue timeout` from `query timeout`, and say which one used resources on the BE;
- the dashboard user ran in its default group and was refused the ad hoc group, and you can say what each of `USAGE_PRIV` and `default_workload_group` does;
- the SQL Block Rule rejected the full scan before it ran, and you can say what its scope was;
- you removed the users, the rule, the role, and the groups in an order that their dependencies allowed, and `normal` is unchanged.

## Next

Open [Quiz 14: Resource Isolation and Query Protection](quiz14_resource_isolation.ipynb). This is the last Lab of the course: the [course README](../../README.md) sums up what you have built. Use [Clean up](../../README.md#clean-up) to pause and resume services with their data, or, after you finish the optional Labs, remove all course resources in the documented dependency order.